In [1]:
import subprocess
from typing import Any, Dict, Optional

import mlflow
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from pandas import DataFrame
from sklearn.base import BaseEstimator
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
random_state = 42
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

def train(
    model: BaseEstimator,
    X_train: DataFrame,
    y_train: DataFrame,
    X_test: DataFrame,
    y_test: DataFrame,
    param_grid: Optional[Dict[str, Any]] = None,
    run_name: Optional[str] = None,
    tags: Optional[Dict[str, str]] = None,
    cv: int = 5,
    scoring: Optional[str] = None,
    n_jobs: Optional[int] = 2
) -> None:
    mlflow.set_experiment("Imo")

    with mlflow.start_run(run_name=run_name, tags=tags):
        mlflow.sklearn.autolog(log_models=False, max_tuning_runs=0)
        if param_grid:
            grid = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                cv=cv,
                scoring=scoring,
                n_jobs=n_jobs
            )
            grid.fit(X_train, y_train)
            best_est = grid.best_estimator_
            mlflow.log_params(grid.best_params_)
            mlflow.set_tag("best_params", str(grid.best_params_))
        else:
            model.fit(X_train, y_train)
            best_est = model
        test_score = best_est.score(X_test, y_test)
        mlflow.log_metric("test_score", test_score)

In [3]:
train(LinearRegression(), X_train, y_train, X_test, y_test, run_name="LinearRegression")
train(LinearRegression(), X_train_scaled, y_train, X_test_scaled, y_test, run_name="LinearRegression_scaled")

In [5]:
params_grid = { "alpha": np.logspace(-3, 3, 10) }
model = { 'Ridge': Ridge(random_state=random_state), 'Lasso': Lasso(random_state=random_state)}
for name, mod in model.items():
    train(mod, X_train, y_train, X_test, y_test, param_grid=params_grid, run_name=name)
    train(mod, X_train_scaled, y_train, X_test_scaled, y_test, param_grid=params_grid, run_name=f"{name}_scaled")

2025/08/24 22:05:14 INFO mlflow.sklearn.utils: Logging no runs, 10 runs will be omitted.
2025/08/24 22:05:15 INFO mlflow.sklearn.utils: Logging no runs, 10 runs will be omitted.
2025/08/24 22:05:18 INFO mlflow.sklearn.utils: Logging no runs, 10 runs will be omitted.
2025/08/24 22:05:20 INFO mlflow.sklearn.utils: Logging no runs, 10 runs will be omitted.


In [6]:
params_grid = { "alpha": np.logspace(-3, 3, 10), "l1_ratio": np.linspace(0.001, 0.999, 10) }
train(ElasticNet(random_state=random_state), X_train, y_train, X_test, y_test, param_grid=params_grid, run_name="ElasticNet")
train(ElasticNet(random_state=random_state), X_train_scaled, y_train, X_test_scaled, y_test, param_grid=params_grid, run_name="ElasticNet_scaled")

2025/08/24 22:05:37 INFO mlflow.sklearn.utils: Logging no runs, 100 runs will be omitted.
2025/08/24 22:05:49 INFO mlflow.sklearn.utils: Logging no runs, 100 runs will be omitted.


In [7]:
params_grid = { "max_depth": [None, 3, 5],
                "n_estimators": [100, 150] }
train(RandomForestRegressor(random_state=random_state), X_train, y_train, X_test, y_test, param_grid=params_grid, run_name="RandomForest")

2025/08/24 22:08:13 INFO mlflow.sklearn.utils: Logging no runs, 6 runs will be omitted.


In [8]:
params_grid = { "max_depth": [None, 3, 5],
                "n_estimators": [100, 150],
                 "learning_rate": [0.1, 0.15] }
train(GradientBoostingRegressor(random_state=random_state), X_train, y_train, X_test, y_test, param_grid=params_grid, run_name="GradientBoosting")

2025/08/24 22:14:35 INFO mlflow.sklearn.utils: Logging no runs, 12 runs will be omitted.


# Résulats

In [9]:
mlflow_ui_process = subprocess.Popen(["mlflow", "ui", "--port", "5000"])
display(Markdown("[➡️ Ouvrir MLflow UI ici](http://localhost:5000)"))

[➡️ Ouvrir MLflow UI ici](http://localhost:5000)

L'erreur quadratique moyenne (MSE) entre les valeurs prédites et les valeurs réelles met en évidence les modèles qui commettent des erreurs importantes, en les pénalisant fortement. À partir de ce graphique, nous pouvons conclure que le modèle avec le meilleur pouvoir prédictif est le Gradient Boosting. De manière générale, il est également observé que les modèles basés sur des arbres surpassent les modèles linéaires dans le cadre de ce projet. De plus, il est intéressant de noter que les modèles linéaires entraînés sur des données standardisées présentent des performances inférieures à ceux entraînés sur les données brutes. 

La moyenne des erreurs absolues (MAE) entre les valeurs réelles et les prédictions se montre plus robuste face aux valeurs extrêmes. À partir de ce graphique, nous pouvons conclure que le modèle le plus résilient face aux valeurs extrêmes est le Gradient Boosting. De manière générale, il est également observé que les modèles basés sur des arbres continuent de surpasser les modèles linéaires dans le cadre de ce projet. Par ailleurs, il est intéressant de noter que, contrairement aux résultats précédents, les modèles linéaires entraînés sur des données standardisées offrent des performances supérieures à ceux entraînés sur des données brutes en termes de robustesse face aux valeurs manquantes.

Le coefficient de détermination r2 reflète la proportion de variance des données réelles expliquée par les modèles prédictifs. À partir de ce graphique, nous constatons un comportement similaire à celui observé avec l'erreur quadratique moyenne (MSE). Le modèle Gradient Boosting se distingue une fois de plus par son meilleur pouvoir explicatif. De manière générale, les modèles basés sur des arbres surpassent nettement les modèles linéaires dans ce projet. Par ailleurs, il est confirmé que les modèles linéaires entraînés sur des données brutes affichent de meilleures performances que ceux entraînés sur des données standardisées, en cohérence avec les résultats obtenus pour la MSE.

**Conclusion :** Notre modèle à mettre en production est le Gradient Boosting.